

### Cell 1: Environment & Loading Models with Your Exact Paths

In [ ]:
import os
import tarfile
import torch
import torch.nn as nn
import random
import pandas as pd
import numpy as np
import nibabel as nib
from tqdm import tqdm
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from huggingface_hub import login

# --- YOUR EXACT KAGGLE INPUT PATHS ---
ADAPTER_PATH = "/kaggle/input/notebooks/axha241419/training-2/brain_tumor_vlm_final/adapter.pt"
LORA_DIR     = "/kaggle/input/notebooks/axha241419/training-2/brain_tumor_vlm_final/lora"
QA_CSV_PATH  = "/kaggle/input/notebooks/axha241419/training-2/brats_clinical_qa_4_features.csv"
TAR_PATH     = "/kaggle/input/notebooks/aliqaiser1123/brats2021-task1-preprocessing/brats2021_processed.tar"

print("========================================")
print("🧠 1. CLEANLY LOADING MODEL WEIGHTS & DATA")
print("========================================")

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 1. Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load Base LLaMA Model
print("⏳ Loading Base LLaMA-3.1 into GPU...")
base_llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token

# 3. Apply LoRA Weights onto LLaMA from your Kaggle Input Path
print("⏳ Applying Trained LoRA Weights...")
llm = PeftModel.from_pretrained(base_llm, LORA_DIR).eval()

# 4. Load 3D Vision Encoder
class BrainIAC3DEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        self.proj = nn.Linear(4, embed_dim)

    def forward(self, x):
        x_pooled = x.mean(dim=[-2, -1]).transpose(1, 2)
        x_resampled = nn.functional.interpolate(x_pooled.transpose(1, 2), size=10, mode='linear').transpose(1, 2)
        return self.proj(x_resampled).to(torch.bfloat16)

print("⏳ Loading 3D Vision Encoder...")
brainiac_encoder = BrainIAC3DEncoder().cuda().bfloat16().eval()

# 5. Build Standalone Vision Adapter & Load Checkpoint
print("⏳ Loading 3D Vision Adapter Weights...")
vision_adapter = nn.Sequential(
    nn.Linear(768, 4096),
    nn.GELU(),
    nn.Linear(4096, 4096),
    nn.GELU(),
    nn.Linear(4096, 4096)
).cuda().bfloat16().eval()

adapter_ckpt = torch.load(ADAPTER_PATH)
clean_ckpt = {k.replace("adapter.", "").replace("proj.", ""): v for k, v in adapter_ckpt.items()}
vision_adapter.load_state_dict(clean_ckpt)

print("✅ Model & Adapter successfully initialized!")

---

### Cell 2: Fixed Streamer, Preprocessing & Prompting Functions

In [ ]:
class BraTSVirtualStreamer:
    def __init__(self, tar_path):
        self.tar_path = tar_path
        self.temp_dir = "/tmp/brats_stream_eval"
        os.makedirs(self.temp_dir, exist_ok=True)

    def stream_patient(self, patient_id):
        if self.tar_path and os.path.exists(self.tar_path):
            with tarfile.open(self.tar_path, 'r') as tar:
                patient_files = [m for m in tar.getmembers() if patient_id in m.name]
                tar.extractall(path=self.temp_dir, members=patient_files)
            yield os.path.join(self.temp_dir, patient_id)
            # Immediate Cleanup
            for f in os.listdir(self.temp_dir):
                file_path = os.path.join(self.temp_dir, f)
                if os.path.isfile(file_path):
                    os.remove(file_path)
        else:
            yield None

def load_patient_3d_volume(patient_dir):
    if not patient_dir or not os.path.exists(patient_dir):
        raise ValueError(f"CRITICAL ERROR: Patient directory {patient_dir} not found!")

    files = [f for f in os.listdir(patient_dir) if f.endswith('.nii.gz') or f.endswith('.npy')]
    if not files:
        raise ValueError(f"CRITICAL ERROR: No MRI scans found in {patient_dir}")

    file_path = os.path.join(patient_dir, files[0])
    img = nib.load(file_path).get_fdata() if file_path.endswith('.nii.gz') else np.load(file_path)
    tensor = torch.from_numpy(img).float().cuda().bfloat16()
    if tensor.ndim == 3:
        tensor = tensor.unsqueeze(0).unsqueeze(0)
    elif tensor.ndim == 4:
        tensor = tensor.unsqueeze(0)
    return tensor

# Initialize streamer globally with the correct path
streamer = BraTSVirtualStreamer(TAR_PATH)

def generate_vlm_answer(patient_id, question):
    """Generates direct clinical answers using real 3D MRI volume + text embeddings."""
    volume_tensor = None
    for patient_dir in streamer.stream_patient(patient_id):
        volume_tensor = load_patient_3d_volume(patient_dir)

    if volume_tensor is None:
        return "Error: Could not load patient volume."

    with torch.no_grad():
        # 1. Vision Embeddings
        image_embs = brainiac_encoder(volume_tensor)
        projected_image = vision_adapter(image_embs).to(dtype=torch.bfloat16)

        # 2. Explicit LLaMA-3.1 Chat System & User Prompt Structure
        formatted_prompt = (
            "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
            "You are an expert medical AI assistant analyzing 3D Brain MRI scans. "
            "Answer the user's clinical question accurately and concisely.<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n{question}<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n"
        )

        text_inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
        text_embeddings = llm.get_base_model().get_input_embeddings()(text_inputs.input_ids).to(dtype=torch.bfloat16)

        # 3. Concatenate Visual Tokens + Text Embeddings
        inputs_embeds = torch.cat([projected_image, text_embeddings], dim=1)

        # 4. Combined Attention Mask
        vision_mask = torch.ones((1, projected_image.shape[1]), device="cuda", dtype=torch.long)
        attention_mask = torch.cat([vision_mask, text_inputs.attention_mask], dim=1)

        # 5. Generate Response
        outputs = llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=40,
            do_sample=False,              # Deterministic execution for evaluation
            repetition_penalty=1.2,       # Prevents digit loops/stuttering
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

        # 6. Extract Response
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        if "assistant" in generated_text:
            generated_text = generated_text.split("assistant")[-1].strip()

        return generated_text.strip()

---

### Cell 3: Evaluation Suite

In [ ]:
print("========================================")
print("👁️ 2. QUALITATIVE DEMO")
print("========================================")

qa_df = pd.read_csv(QA_CSV_PATH)
demo_samples = qa_df.sample(3, random_state=7)

for idx, row in demo_samples.iterrows():
    p_id = row['patient_id']
    q = row['question']
    real_a = row['answer']

    print(f"\n🩺 Patient: {p_id}")
    print(f"❓ Question: {q}")
    print(f"✅ Ground Truth: {real_a}")

    pred_a = generate_vlm_answer(p_id, q)
    print(f"🤖 Model Output: {pred_a}")

print("\n========================================")
print("📊 3. QUANTITATIVE EVALUATION SUITE")
print("========================================")

eval_samples = qa_df.sample(50, random_state=42)
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smoother = SmoothingFunction().method1

total_bleu = 0
total_rouge = 0
exact_match_count = 0

for idx, row in tqdm(eval_samples.iterrows(), total=len(eval_samples), desc="Evaluating"):
    q = row['question']
    real_a = row['answer']
    pred_a = generate_vlm_answer(row['patient_id'], q)

    # BLEU-4
    bleu = sentence_bleu([real_a.split()], pred_a.split(), smoothing_function=smoother)
    total_bleu += bleu

    # ROUGE-L
    rouge_score = rouge.score(real_a, pred_a)['rougeL'].fmeasure
    total_rouge += rouge_score

    # Heuristic Match Check
    if ("unifocal" in real_a.lower() and "unifocal" in pred_a.lower()) or \
       ("multifocal" in real_a.lower() and "multifocal" in pred_a.lower()) or \
       (rouge_score > 0.60):
        exact_match_count += 1

print("\n========================================")
print("🏆 FINAL EVALUATION METRICS")
print("========================================")
print(f"🔹 Average BLEU-4 Score : {total_bleu / len(eval_samples):.4f}")
print(f"🔹 Average ROUGE-L Score: {total_rouge / len(eval_samples):.4f}")
print(f"🔹 Clinical Accuracy    : {(exact_match_count / len(eval_samples)) * 100:.2f}%")
print("========================================")